In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ============================================================
# 06_backbone_benchmark_development_only.ipynb
# COMPLETE SINGLE-CELL CODE
# ============================================================
#
# Purpose
# -------
# Perform a CLEAN, UNIFORM development-only comparison of the
# two backbones that have already been independently selected
# and confirmed without using the official test set:
#
#   ViT:
#       expected ordinal-distance
#       alpha = 1
#       lambda = 0.20
#
#   Swin:
#       expected ordinal-distance
#       alpha = 1
#       lambda = 0.10
#
# Both backbones are retrained under exactly the same:
#   - 478 development patients
#   - frozen five patient-stratified folds
#   - modal_low patient reference
#   - mean_probability_argmax aggregation
#   - seeds 42, 2025, 2026
#   - image preprocessing / augmentation
#   - optimizer and LR schedule
#   - class weighting
#   - TTA
#   - checkpoint hierarchy
#   - runtime worker configuration
#
# Primary comparison
# ------------------
# Mean pooled OOF patient-level QWK across seeds.
#
# Secondary hierarchy
# -------------------
# 1. Patient MAE
# 2. Patient macro F1
# 3. Patient accuracy
# 4. Image QWK
#
# Test policy
# -----------
# The official LIMUC test partition is NEVER:
#   - defined
#   - loaded
#   - evaluated
#   - used for model selection
#
# Runtime
# -------
# Google Drive image access:
#   num_workers = 0
#   local /content image cache
#
# Resume
# ------
# - completed seeds are reused
# - completed folds are reused
# - interrupted folds resume after the last completed epoch
#
# Workload
# --------
# 2 backbones x 3 seeds x 5 folds = 30 fold trainings
# ============================================================


# ============================================================
# 0. Install dependencies
# ============================================================

%pip -q install timm==1.0.28 scikit-learn==1.6.1 pandas==2.2.2 Pillow==11.3.0


# ============================================================
# 1. Imports
# ============================================================

import gc
import io
import os
import json
import math
import copy
import time
import random
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.metrics import (
    cohen_kappa_score,
    mean_absolute_error,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

from IPython.display import display


ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")


# ============================================================
# 2. Development-only safety check
# ============================================================

FORBIDDEN_TEST_OBJECTS = [
    "TEST_PATH",
    "TEST_ROOT",
    "TEST_DIR",
    "OFFICIAL_TEST_ROOT",
    "test_df",
    "test_dataset",
    "test_loader",
    "test_predictions",
    "test_patient_predictions",
    "test_fold_prob_dfs",
    "all_test_fold_prob_df",
    "ensemble_test_df",
    "ensemble_test_patient_df",
    "ensemble_test_metrics",
    "ensemble_test_patient_metrics",
]

present_test_objects = [
    name
    for name in FORBIDDEN_TEST_OBJECTS
    if name in globals()
]

if present_test_objects:
    raise RuntimeError(
        "Official-test objects exist in the current runtime: "
        f"{present_test_objects}\n"
        "Restart the runtime and run only this cell."
    )

print("Development-only safety check passed.")
print("Official LIMUC test data are not defined or loaded.")


# ============================================================
# 3. Mount Google Drive
# ============================================================

from google.colab import drive

(drive.mount(
    "/content/drive",
    force_remount=True,
) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)


# ============================================================
# 4. Paths
# ============================================================

DEVELOPMENT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/train_and_validation_sets')
)

VIT_SCREENING_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/01_vit_loss_screening_development_only')
)

VIT_CONFIRMATION_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/02_vit_loss_multiseed_confirmation_epoch_resume_v2')
)

AGGREGATION_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/03_patient_aggregation_and_reference_sensitivity')
)

SWIN_SCREENING_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/04_swin_loss_screening_development_only')
)

SWIN_CONFIRMATION_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/05_swin_loss_multiseed_confirmation_uniform_runtime_v1')
)

OUTPUT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/06_backbone_benchmark_development_only_v1')
)

MANIFEST_PATH = (
    VIT_SCREENING_ROOT
    / "development_image_manifest_with_folds.csv"
)

FOLD_ASSIGNMENT_PATH = (
    VIT_SCREENING_ROOT
    / "patient_fold_assignment.csv"
)

REFERENCE_AUDIT_PATH = (
    VIT_SCREENING_ROOT
    / "patient_reference_rule_sensitivity.csv"
)

VIT_SCREENING_PLAN_PATH = (
    VIT_SCREENING_ROOT
    / "analysis_plan_frozen_before_training.json"
)

AGGREGATION_PLAN_PATH = (
    AGGREGATION_ROOT
    / "analysis_plan_frozen_before_aggregation.json"
)

LOCKED_AGGREGATION_PATH = (
    AGGREGATION_ROOT
    / "locked_patient_aggregation_rule.json"
)

SWIN_CONFIRMATION_PLAN_PATH = (
    SWIN_CONFIRMATION_ROOT
    / "analysis_plan_frozen_before_confirmation.json"
)

LOCKED_SWIN_PATH = (
    SWIN_CONFIRMATION_ROOT
    / "locked_representative_swin_configuration.json"
)

required_paths = [
    DEVELOPMENT_ROOT,
    VIT_SCREENING_ROOT,
    VIT_CONFIRMATION_ROOT,
    AGGREGATION_ROOT,
    SWIN_CONFIRMATION_ROOT,
    MANIFEST_PATH,
    FOLD_ASSIGNMENT_PATH,
    REFERENCE_AUDIT_PATH,
    VIT_SCREENING_PLAN_PATH,
    AGGREGATION_PLAN_PATH,
    LOCKED_AGGREGATION_PATH,
    SWIN_CONFIRMATION_PLAN_PATH,
    LOCKED_SWIN_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path does not exist:\n{required_path}"
        )

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nDevelopment root:", DEVELOPMENT_ROOT)
print("ViT development files:", VIT_SCREENING_ROOT)
print("Swin confirmation files:", SWIN_CONFIRMATION_ROOT)
print("Backbone benchmark output:", OUTPUT_ROOT)


# ============================================================
# 5. Helper functions for reading frozen plans
# ============================================================

def recursively_collect_strings(
    object_value,
):
    collected = []

    if isinstance(object_value, dict):
        for value in object_value.values():
            collected.extend(
                recursively_collect_strings(value)
            )

    elif isinstance(object_value, (list, tuple)):
        for value in object_value:
            collected.extend(
                recursively_collect_strings(value)
            )

    elif isinstance(object_value, str):
        collected.append(object_value)

    return collected


def resolve_vit_model_name_from_plan(
    plan,
):
    """
    Find the actual timm ViT model identifier that was used
    in Notebook 01.

    Candidate experiment names such as vit_L5_... are ignored
    because their base names are not registered timm models.
    """

    all_strings = recursively_collect_strings(
        plan
    )

    base_registry = set(
        timm.list_models()
    )

    pretrained_registry = set(
        timm.list_models(
            pretrained=True
        )
    )

    candidate_strings = []

    for value in all_strings:
        value = str(value)

        if not value.startswith("vit_"):
            continue

        base_name = value.split(
            ".",
            1,
        )[0]

        if (
            base_name in base_registry
            or
            value in pretrained_registry
        ):
            candidate_strings.append(
                value
            )

    if candidate_strings:
        candidate_strings = sorted(
            set(candidate_strings),
            key=lambda value: (
                "384" not in value,
                "patch16" not in value,
                "." not in value,
                value,
            ),
        )

        return candidate_strings[0]

    # Deterministic fallback only if the frozen plan does not
    # expose the model identifier in a searchable field.
    fallback_candidates = [
        "vit_base_patch16_384.augreg_in21k_ft_in1k",
        "vit_base_patch16_384.augreg_in1k",
        "vit_base_patch16_384",
    ]

    for candidate in fallback_candidates:
        try:
            temporary_model = timm.create_model(
                candidate,
                pretrained=False,
                num_classes=4,
            )

            del temporary_model
            gc.collect()

            return candidate

        except Exception:
            continue

    raise RuntimeError(
        "Could not resolve the ViT model identifier from "
        "the frozen Notebook 01 plan."
    )


# ============================================================
# 6. Load frozen decisions
# ============================================================

with open(
    VIT_SCREENING_PLAN_PATH,
    "r",
    encoding="utf-8",
) as file:
    vit_screening_plan = json.load(
        file
    )

with open(
    SWIN_CONFIRMATION_PLAN_PATH,
    "r",
    encoding="utf-8",
) as file:
    swin_confirmation_plan = json.load(
        file
    )

with open(
    LOCKED_SWIN_PATH,
    "r",
    encoding="utf-8",
) as file:
    locked_swin_record = json.load(
        file
    )

with open(
    LOCKED_AGGREGATION_PATH,
    "r",
    encoding="utf-8",
) as file:
    locked_aggregation_record = json.load(
        file
    )


PRIMARY_REFERENCE_RULE = "modal_low"

LOCKED_AGGREGATION_RULE = (
    "mean_probability_argmax"
)

if (
    locked_aggregation_record.get(
        "locked_aggregation_rule"
    )
    !=
    LOCKED_AGGREGATION_RULE
):
    raise RuntimeError(
        "Notebook 03 aggregation lock differs from "
        "mean_probability_argmax."
    )

if (
    locked_aggregation_record.get(
        "primary_reference_rule"
    )
    !=
    PRIMARY_REFERENCE_RULE
):
    raise RuntimeError(
        "Notebook 03 primary reference differs "
        "from modal_low."
    )


VIT_MODEL_NAME = (
    resolve_vit_model_name_from_plan(
        vit_screening_plan
    )
)

SWIN_MODEL_NAME = str(
    swin_confirmation_plan[
        "resolved_model_name"
    ]
)

if (
    locked_swin_record.get(
        "representative_swin_candidate"
    )
    !=
    "swin_L3_expected_ordinal_distance_a1_w010"
):
    raise RuntimeError(
        "Notebook 05 did not lock the expected "
        "representative Swin configuration."
    )


# ============================================================
# 7. Frozen representative backbone configurations
# ============================================================

BACKBONE_CONFIGS = {
    "ViT": {
        "backbone_name": (
            "ViT"
        ),

        "model_name": (
            VIT_MODEL_NAME
        ),

        "representative_configuration": (
            "vit_L5_expected_ordinal_distance_a1_w020"
        ),

        "loss_type": (
            "ce_expected_distance"
        ),

        "ordinal_alpha": 1.0,

        "ordinal_loss_weight": 0.20,
    },

    "Swin": {
        "backbone_name": (
            "Swin"
        ),

        "model_name": (
            SWIN_MODEL_NAME
        ),

        "representative_configuration": (
            "swin_L3_expected_ordinal_distance_a1_w010"
        ),

        "loss_type": (
            "ce_expected_distance"
        ),

        "ordinal_alpha": 1.0,

        "ordinal_loss_weight": 0.10,
    },
}

BACKBONES_TO_RUN = [
    "ViT",
    "Swin",
]

CONFIRMATION_SEEDS = [
    42,
    2025,
    2026,
]

print(
    "\nVerified primary patient reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Verified patient aggregation:",
    LOCKED_AGGREGATION_RULE,
)

print(
    "Resolved ViT model:",
    VIT_MODEL_NAME,
)

print(
    "Resolved Swin model:",
    SWIN_MODEL_NAME,
)

print(
    "Benchmark seeds:",
    CONFIRMATION_SEEDS,
)


# ============================================================
# 8. Uniform scientific configuration
# ============================================================

PROTOCOL_VERSION = (
    "uniform_vit_swin_backbone_benchmark_v1"
)

NUM_CLASSES = 4
NUM_FOLDS = 5

IMAGE_SIZE = 384

# Uniform implementation across both architectures.
TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2

EFFECTIVE_TRAIN_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

VALIDATION_BATCH_SIZE = 8

RUNTIME_NUM_WORKERS = 0

EPOCHS = 25

LEARNING_RATE = 1e-4
MIN_LR_RATIO = 0.05
WARMUP_EPOCHS = 3
WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.02

DROP_RATE = 0.10
DROP_PATH_RATE = 0.05

EARLY_STOP_PATIENCE = 8

GRAD_CLIP_NORM = 1.0

USE_MIXED_PRECISION = True

USE_CLASS_WEIGHTS_IN_CE = True

USE_TTA = True
TTA_HORIZONTAL_FLIP = True

BOOTSTRAP_REPETITIONS = 5000
BOOTSTRAP_SEED = 20260806

EPOCH_CHECKPOINT_INTERVAL = 1

RESUME_COMPLETED_FOLDS = True
RESUME_INTERRUPTED_FOLDS = True
SKIP_COMPLETED_SEED_EXPERIMENTS = True


# ============================================================
# 9. Google Drive I/O safeguards
# ============================================================

USE_LOCAL_IMAGE_CACHE = True

LOCAL_IMAGE_CACHE_ROOT = Path(
    "/content/LIMUC_dev_cache_backbone06"
)

LOCAL_IMAGE_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

IMAGE_IO_MAX_RETRIES = 12
IMAGE_IO_INITIAL_DELAY_SECONDS = 1.0
IMAGE_IO_MAX_DELAY_SECONDS = 8.0

DRIVE_IO_MAX_RETRIES = 5
DRIVE_IO_INITIAL_DELAY_SECONDS = 1.0


# ============================================================
# 10. Atomic file helpers
# ============================================================

def temporary_path(
    path: Path,
):
    return path.with_name(
        path.name + ".temporary"
    )


def retry_delay(
    attempt_index,
    initial_delay,
    maximum_delay=8.0,
):
    return min(
        maximum_delay,
        initial_delay
        * (attempt_index + 1),
    )


def atomic_json_save(
    payload,
    path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        temp_path = temporary_path(
            path
        )

        try:
            if temp_path.exists():
                temp_path.unlink()

            with open(
                temp_path,
                "w",
                encoding="utf-8",
            ) as file:
                json.dump(
                    payload,
                    file,
                    indent=2,
                    default=str,
                )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save JSON:\n{path}\n"
        f"{repr(last_error)}"
    )


def atomic_csv_save(
    dataframe,
    path,
    index=False,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        temp_path = temporary_path(
            path
        )

        try:
            if temp_path.exists():
                temp_path.unlink()

            dataframe.to_csv(
                temp_path,
                index=index,
            )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            print(
                "CSV write retry:",
                attempt_index + 1,
                path,
            )

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save CSV:\n{path}\n"
        f"{repr(last_error)}"
    )


def atomic_torch_save(
    payload,
    path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        temp_path = temporary_path(
            path
        )

        try:
            if temp_path.exists():
                temp_path.unlink()

            torch.save(
                payload,
                temp_path,
            )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            print(
                "Checkpoint write retry:",
                attempt_index + 1,
                path,
            )

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save checkpoint:\n{path}\n"
        f"{repr(last_error)}"
    )


def safe_torch_load(
    path,
    map_location="cpu",
):
    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        try:
            try:
                return torch.load(
                    path,
                    map_location=map_location,
                    weights_only=False,
                )

            except TypeError:
                return torch.load(
                    path,
                    map_location=map_location,
                )

        except OSError as error:
            last_error = error

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not load checkpoint:\n{path}\n"
        f"{repr(last_error)}"
    )


for abandoned_file in OUTPUT_ROOT.rglob(
    "*.temporary"
):
    try:
        abandoned_file.unlink()
    except Exception:
        pass


# ============================================================
# 11. Hash / environment helpers
# ============================================================

def file_sha256(
    path,
):
    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            hasher.update(
                block
            )

    return hasher.hexdigest()


def dictionary_sha256(
    dictionary,
):
    encoded = json.dumps(
        dictionary,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


def safe_package_version(
    package_name,
):
    try:
        return version(
            package_name
        )

    except PackageNotFoundError:
        return None


# ============================================================
# 12. RNG helpers
# ============================================================

def seed_everything(
    seed,
):
    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(
            seed
        )


def seed_worker(
    worker_id,
):
    worker_seed = (
        torch.initial_seed()
        %
        (2**32)
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


def as_cpu_byte_tensor(
    state,
    name,
):
    if isinstance(
        state,
        torch.Tensor,
    ):
        return (
            state
            .detach()
            .to(
                "cpu",
                dtype=torch.uint8,
            )
            .contiguous()
        )

    if isinstance(
        state,
        np.ndarray,
    ):
        return (
            torch.from_numpy(
                state.astype(
                    np.uint8,
                    copy=False,
                )
            )
            .cpu()
            .contiguous()
        )

    return torch.tensor(
        state,
        dtype=torch.uint8,
        device="cpu",
    ).contiguous()


def capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),

        "torch_cpu": (
            torch.get_rng_state()
            .cpu()
            .contiguous()
        ),

        "torch_cuda": None,
    }

    if torch.cuda.is_available():
        state["torch_cuda"] = [
            cuda_state.cpu().contiguous()
            for cuda_state in (
                torch.cuda.get_rng_state_all()
            )
        ]

    return state


def restore_rng_state(
    state,
):
    random.setstate(
        state["python"]
    )

    numpy_state = state["numpy"]

    if isinstance(
        numpy_state,
        list,
    ):
        numpy_state = tuple(
            numpy_state
        )

    np.random.set_state(
        numpy_state
    )

    torch.set_rng_state(
        as_cpu_byte_tensor(
            state["torch_cpu"],
            "CPU RNG",
        )
    )

    if (
        torch.cuda.is_available()
        and
        state.get("torch_cuda")
        is not None
    ):
        cuda_states = [
            as_cpu_byte_tensor(
                value,
                f"CUDA RNG {index}",
            )
            for index, value in enumerate(
                state["torch_cuda"]
            )
        ]

        if (
            len(cuda_states)
            !=
            torch.cuda.device_count()
        ):
            if (
                torch.cuda.device_count()
                ==
                1
                and
                len(cuda_states)
                >=
                1
            ):
                cuda_states = [
                    cuda_states[0]
                ]

            else:
                raise RuntimeError(
                    "CUDA RNG-state count mismatch."
                )

        torch.cuda.set_rng_state_all(
            cuda_states
        )


def optimizer_to_device(
    optimizer,
    target_device,
):
    for optimizer_state in (
        optimizer.state.values()
    ):
        for key, value in list(
            optimizer_state.items()
        ):
            if isinstance(
                value,
                torch.Tensor,
            ):
                optimizer_state[
                    key
                ] = value.to(
                    target_device
                )


seed_everything(
    CONFIRMATION_SEEDS[0]
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.set_float32_matmul_precision(
        "high"
    )
except Exception:
    pass


# ============================================================
# 13. Verify both timm model identifiers
# ============================================================

def verify_timm_model_identifier(
    model_name,
):
    base_model_name = (
        model_name
        .split(
            ".",
            1,
        )[0]
    )

    base_registry = set(
        timm.list_models()
    )

    if (
        base_model_name
        not in
        base_registry
    ):
        raise RuntimeError(
            f"Base timm model is unavailable: "
            f"{base_model_name}"
        )

    try:
        verification_model = (
            timm.create_model(
                model_name,
                pretrained=False,
                num_classes=NUM_CLASSES,
                drop_rate=DROP_RATE,
                drop_path_rate=(
                    DROP_PATH_RATE
                ),
            )
        )

        del verification_model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(
            "Verified timm identifier:",
            model_name,
        )

    except Exception as error:
        raise RuntimeError(
            f"Could not instantiate timm model:\n"
            f"{model_name}\n"
            f"{repr(error)}"
        ) from error


for backbone_name in BACKBONES_TO_RUN:
    verify_timm_model_identifier(
        BACKBONE_CONFIGS[
            backbone_name
        ]["model_name"]
    )


# ============================================================
# 14. Freeze benchmark analysis plan
# ============================================================

BENCHMARK_TRAINING_CONFIG = {
    "num_classes": NUM_CLASSES,

    "num_folds": NUM_FOLDS,

    "image_size": IMAGE_SIZE,

    "train_batch_size": (
        TRAIN_BATCH_SIZE
    ),

    "gradient_accumulation_steps": (
        GRADIENT_ACCUMULATION_STEPS
    ),

    "effective_train_batch_size": (
        EFFECTIVE_TRAIN_BATCH_SIZE
    ),

    "validation_batch_size": (
        VALIDATION_BATCH_SIZE
    ),

    "runtime_num_workers": (
        RUNTIME_NUM_WORKERS
    ),

    "epochs": EPOCHS,

    "learning_rate": (
        LEARNING_RATE
    ),

    "warmup_epochs": (
        WARMUP_EPOCHS
    ),

    "min_lr_ratio": (
        MIN_LR_RATIO
    ),

    "weight_decay": (
        WEIGHT_DECAY
    ),

    "label_smoothing": (
        LABEL_SMOOTHING
    ),

    "drop_rate": (
        DROP_RATE
    ),

    "drop_path_rate": (
        DROP_PATH_RATE
    ),

    "early_stop_patience": (
        EARLY_STOP_PATIENCE
    ),

    "grad_clip_norm": (
        GRAD_CLIP_NORM
    ),

    "use_mixed_precision": (
        USE_MIXED_PRECISION
    ),

    "use_class_weights_in_ce": (
        USE_CLASS_WEIGHTS_IN_CE
    ),

    "use_tta": (
        USE_TTA
    ),

    "tta_horizontal_flip": (
        TTA_HORIZONTAL_FLIP
    ),

    "primary_patient_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "patient_aggregation": (
        LOCKED_AGGREGATION_RULE
    ),
}

immutable_analysis_plan = {
    "protocol_version": (
        PROTOCOL_VERSION
    ),

    "study_stage": (
        "Uniform development-only ViT vs Swin "
        "backbone benchmark"
    ),

    "official_test_loaded": False,

    "official_test_evaluated": False,

    "backbone_configurations": (
        BACKBONE_CONFIGS
    ),

    "training_seeds": (
        CONFIRMATION_SEEDS
    ),

    "training_config": (
        BENCHMARK_TRAINING_CONFIG
    ),

    "primary_ranking_rule": (
        "Mean pooled OOF patient QWK "
        "across three seeds, higher is better"
    ),

    "secondary_ranking_hierarchy": [
        "Mean pooled OOF patient MAE: lower",
        "Mean pooled OOF patient macro F1: higher",
        "Mean pooled OOF patient accuracy: higher",
        "Mean pooled OOF image QWK: higher",
    ],

    "checkpoint_selection_hierarchy": [
        "Validation patient QWK: higher",
        "Validation patient MAE: lower",
        "Validation patient macro F1: higher",
        "Validation image QWK: higher",
    ],

    "development_manifest_sha256": (
        file_sha256(
            MANIFEST_PATH
        )
    ),

    "fold_assignment_sha256": (
        file_sha256(
            FOLD_ASSIGNMENT_PATH
        )
    ),

    "reference_audit_sha256": (
        file_sha256(
            REFERENCE_AUDIT_PATH
        )
    ),

    "locked_aggregation_sha256": (
        file_sha256(
            LOCKED_AGGREGATION_PATH
        )
    ),

    "vit_screening_plan_sha256": (
        file_sha256(
            VIT_SCREENING_PLAN_PATH
        )
    ),

    "swin_confirmation_plan_sha256": (
        file_sha256(
            SWIN_CONFIRMATION_PLAN_PATH
        )
    ),

    "locked_swin_sha256": (
        file_sha256(
            LOCKED_SWIN_PATH
        )
    ),

    "bootstrap_repetitions": (
        BOOTSTRAP_REPETITIONS
    ),

    "bootstrap_seed": (
        BOOTSTRAP_SEED
    ),

    "interpretation": (
        "This benchmark compares only backbone-loss "
        "configurations that had already been selected "
        "using development data. No test-set information "
        "is used."
    ),
}

analysis_plan_fingerprint = (
    dictionary_sha256(
        immutable_analysis_plan
    )
)

analysis_plan = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    **immutable_analysis_plan,
}

analysis_plan_path = (
    OUTPUT_ROOT
    / "analysis_plan_frozen_before_backbone_benchmark.json"
)

if analysis_plan_path.exists():
    with open(
        analysis_plan_path,
        "r",
        encoding="utf-8",
    ) as file:
        existing_plan = json.load(
            file
        )

    if (
        existing_plan.get(
            "analysis_plan_fingerprint"
        )
        !=
        analysis_plan_fingerprint
    ):
        raise RuntimeError(
            "Existing Notebook 06 analysis plan differs "
            "from the current configuration.\n"
            "Use a new versioned OUTPUT_ROOT."
        )

    print(
        "\nExisting benchmark plan validated:",
        analysis_plan_path,
    )

else:
    atomic_json_save(
        analysis_plan,
        analysis_plan_path,
    )

    print(
        "\nSaved frozen benchmark plan:",
        analysis_plan_path,
    )

print(
    "Benchmark fingerprint:",
    analysis_plan_fingerprint,
)


# ============================================================
# 15. Environment metadata
# ============================================================

environment_metadata = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "python": (
        platform.python_version()
    ),

    "torch": (
        torch.__version__
    ),

    "torchvision": (
        torchvision.__version__
    ),

    "timm": (
        timm.__version__
    ),

    "numpy": (
        np.__version__
    ),

    "pandas": (
        pd.__version__
    ),

    "scikit_learn": (
        safe_package_version(
            "scikit-learn"
        )
    ),

    "pillow": (
        safe_package_version(
            "Pillow"
        )
    ),

    "cuda_available": bool(
        torch.cuda.is_available()
    ),

    "cuda_version": (
        torch.version.cuda
    ),

    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "official_test_loaded": False,

    "official_test_evaluated": False,
}

atomic_json_save(
    environment_metadata,
    OUTPUT_ROOT
    / "environment_metadata.json",
)


# ============================================================
# 16. Load and validate development manifest
# ============================================================

trainval_df = pd.read_csv(
    MANIFEST_PATH
)

required_manifest_columns = {
    "relative_path",
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
}

missing_manifest_columns = (
    required_manifest_columns
    -
    set(trainval_df.columns)
)

if missing_manifest_columns:
    raise RuntimeError(
        "Manifest missing columns: "
        f"{sorted(missing_manifest_columns)}"
    )

trainval_df["path"] = (
    trainval_df[
        "relative_path"
    ]
    .map(
        lambda relative_path: str(
            DEVELOPMENT_ROOT
            /
            Path(relative_path)
        )
    )
)

for integer_column in [
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
]:
    trainval_df[
        integer_column
    ] = (
        trainval_df[
            integer_column
        ]
        .astype(int)
    )

trainval_df = (
    trainval_df
    .sort_values(
        [
            "patient_id",
            "relative_path",
        ]
    )
    .reset_index(
        drop=True
    )
)

if len(trainval_df) != 9590:
    raise RuntimeError(
        f"Expected 9590 images; "
        f"found {len(trainval_df)}."
    )

if (
    trainval_df[
        "patient_id"
    ].nunique()
    !=
    478
):
    raise RuntimeError(
        "Expected 478 development patients."
    )

if (
    set(
        trainval_df[
            "cv_fold"
        ].unique()
    )
    !=
    set(
        range(
            NUM_FOLDS
        )
    )
):
    raise RuntimeError(
        "Frozen fold identifiers are invalid."
    )


# ============================================================
# 17. Validate modal_low reference
# ============================================================

reference_audit_df = pd.read_csv(
    REFERENCE_AUDIT_PATH
)

reference_audit_df[
    "patient_id"
] = (
    reference_audit_df[
        "patient_id"
    ].astype(int)
)

reference_audit_df[
    PRIMARY_REFERENCE_RULE
] = (
    reference_audit_df[
        PRIMARY_REFERENCE_RULE
    ].astype(int)
)

manifest_patient_df = (
    trainval_df[
        [
            "patient_id",
            "patient_y_true",
            "cv_fold",
        ]
    ]
    .drop_duplicates()
)

reference_check_df = (
    manifest_patient_df
    .merge(
        reference_audit_df[
            [
                "patient_id",
                PRIMARY_REFERENCE_RULE,
            ]
        ],

        on="patient_id",

        how="inner",

        validate="one_to_one",
    )
)

if len(reference_check_df) != 478:
    raise RuntimeError(
        "Patient reference alignment failure."
    )

if not (
    reference_check_df[
        "patient_y_true"
    ]
    ==
    reference_check_df[
        PRIMARY_REFERENCE_RULE
    ]
).all():
    raise RuntimeError(
        "Manifest patient reference differs "
        "from modal_low."
    )

print(
    "\nDevelopment images:",
    len(trainval_df),
)

print(
    "Development patients:",
    trainval_df[
        "patient_id"
    ].nunique(),
)

print(
    "Fold patient counts:",
    manifest_patient_df[
        "cv_fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict(),
)


# ============================================================
# 18. Robust local image cache
# ============================================================

def cache_path_for_relative_path(
    relative_path,
):
    return (
        LOCAL_IMAGE_CACHE_ROOT
        /
        Path(
            str(relative_path)
        )
    )


def read_drive_bytes(
    source_path,
):
    source_path = str(
        source_path
    )

    last_error = None

    for attempt_index in range(
        IMAGE_IO_MAX_RETRIES
    ):
        try:
            with open(
                source_path,
                "rb",
            ) as file:
                data = file.read()

            if len(data) == 0:
                raise OSError(
                    "Zero-byte image read."
                )

            return data

        except (
            OSError,
            IOError,
        ) as error:
            last_error = error

            if (
                attempt_index
                ==
                IMAGE_IO_MAX_RETRIES - 1
            ):
                break

            delay = retry_delay(
                attempt_index,
                IMAGE_IO_INITIAL_DELAY_SECONDS,
                IMAGE_IO_MAX_DELAY_SECONDS,
            )

            print(
                "\nTransient Drive read failure:",
                source_path,
            )

            print(
                "Error:",
                repr(error),
            )

            print(
                f"Retrying in {delay:.1f}s"
            )

            time.sleep(
                delay
            )

    raise OSError(
        f"Unable to read image:\n{source_path}\n"
        f"{repr(last_error)}"
    )


def ensure_cached_image(
    source_path,
    relative_path,
):
    local_path = (
        cache_path_for_relative_path(
            relative_path
        )
    )

    if (
        local_path.exists()
        and
        local_path.stat().st_size > 0
    ):
        return local_path

    local_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    image_bytes = read_drive_bytes(
        source_path
    )

    temp_path = temporary_path(
        local_path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "wb",
    ) as file:
        file.write(
            image_bytes
        )

    os.replace(
        temp_path,
        local_path,
    )

    return local_path


def load_rgb_image(
    source_path,
    relative_path,
):
    local_path = None
    last_error = None

    for attempt_index in range(
        IMAGE_IO_MAX_RETRIES
    ):
        try:
            if USE_LOCAL_IMAGE_CACHE:
                local_path = (
                    ensure_cached_image(
                        source_path,
                        relative_path,
                    )
                )

                read_path = local_path

            else:
                read_path = Path(
                    source_path
                )

            with open(
                read_path,
                "rb",
            ) as file:
                image_bytes = (
                    file.read()
                )

            with Image.open(
                io.BytesIO(
                    image_bytes
                )
            ) as image:
                image.load()

                return image.convert(
                    "RGB"
                )

        except (
            OSError,
            IOError,
            ValueError,
        ) as error:
            last_error = error

            if (
                local_path is not None
                and
                local_path.exists()
            ):
                try:
                    local_path.unlink()
                except Exception:
                    pass

            if (
                attempt_index
                ==
                IMAGE_IO_MAX_RETRIES - 1
            ):
                break

            time.sleep(
                retry_delay(
                    attempt_index,
                    IMAGE_IO_INITIAL_DELAY_SECONDS,
                    IMAGE_IO_MAX_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Unable to decode image:\n{source_path}\n"
        f"{repr(last_error)}"
    )


# ============================================================
# 19. Dataset and transforms
# ============================================================

class LIMUCDataset(Dataset):
    def __init__(
        self,
        dataframe,
        transform=None,
    ):
        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.transform = transform

    def __len__(self):
        return len(
            self.df
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.df.iloc[
            index
        ]

        image = load_rgb_image(
            row["path"],
            row["relative_path"],
        )

        if self.transform is not None:
            image = self.transform(
                image
            )

        return {
            "image": image,

            "label": torch.tensor(
                int(
                    row["label"]
                ),
                dtype=torch.long,
            ),

            "patient_id": torch.tensor(
                int(
                    row["patient_id"]
                ),
                dtype=torch.long,
            ),

            "patient_y_true": torch.tensor(
                int(
                    row[
                        "patient_y_true"
                    ]
                ),
                dtype=torch.long,
            ),

            "path": str(
                row["path"]
            ),
        }


def build_train_transform():
    return transforms.Compose([
        transforms.RandomResizedCrop(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),

            scale=(
                0.75,
                1.00,
            ),

            ratio=(
                0.90,
                1.10,
            ),
        ),

        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        transforms.RandomRotation(
            degrees=8
        ),

        transforms.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.08,
            hue=0.02,
        ),

        transforms.GaussianBlur(
            kernel_size=3,

            sigma=(
                0.1,
                1.2,
            ),
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],

            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


def build_eval_transform():
    return transforms.Compose([
        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],

            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


# ============================================================
# 20. Metrics
# ============================================================

PROBABILITY_COLUMNS = [
    "prob_0",
    "prob_1",
    "prob_2",
    "prob_3",
]


def compute_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    qwk = cohen_kappa_score(
        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
        ],

        weights="quadratic",
    )

    return {
        "qwk": (
            float(qwk)
            if np.isfinite(qwk)
            else np.nan
        ),

        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),

        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,

                labels=[
                    0,
                    1,
                    2,
                    3,
                ],

                average="macro",

                zero_division=0,
            )
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
    }


def make_confusion_df(
    y_true,
    y_pred,
):
    return pd.DataFrame(
        confusion_matrix(
            y_true,
            y_pred,

            labels=[
                0,
                1,
                2,
                3,
            ],
        ),

        index=[
            "true_0",
            "true_1",
            "true_2",
            "true_3",
        ],

        columns=[
            "pred_0",
            "pred_1",
            "pred_2",
            "pred_3",
        ],
    )


# ============================================================
# 21. Locked patient aggregation
# ============================================================

def aggregate_patient_probabilities(
    image_df,
):
    patient_probability_df = (
        image_df
        .groupby(
            "patient_id",
            as_index=False,
        )[PROBABILITY_COLUMNS]
        .mean()
    )

    patient_metadata_df = (
        image_df
        .groupby(
            "patient_id",
            as_index=False,
        )
        .agg(
            y_true=(
                "patient_y_true",
                "first",
            ),

            n_images=(
                "path",
                "size",
            ),
        )
    )

    patient_df = (
        patient_metadata_df
        .merge(
            patient_probability_df,

            on="patient_id",

            how="inner",

            validate="one_to_one",
        )
    )

    patient_df["y_pred"] = (
        patient_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy()
        .argmax(
            axis=1
        )
        .astype(int)
    )

    return (
        patient_df
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 22. Loss
# ============================================================

class ExpectedOrdinalDistanceLoss(
    nn.Module
):
    def __init__(
        self,
        num_classes,
        alpha,
    ):
        super().__init__()

        self.alpha = float(
            alpha
        )

        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )

    def forward(
        self,
        logits,
        targets,
    ):
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        target_values = (
            targets.float()
            .unsqueeze(1)
        )

        distances = torch.abs(
            target_values
            -
            self.class_values.unsqueeze(
                0
            )
        ) ** self.alpha

        return (
            probabilities
            *
            distances
        ).sum(
            dim=1
        ).mean()


class CombinedLoss(nn.Module):
    def __init__(
        self,
        ce_weight,
        ordinal_alpha,
        ordinal_weight,
    ):
        super().__init__()

        self.cross_entropy = (
            nn.CrossEntropyLoss(
                weight=ce_weight,

                label_smoothing=(
                    LABEL_SMOOTHING
                ),
            )
        )

        self.ordinal = (
            ExpectedOrdinalDistanceLoss(
                NUM_CLASSES,
                ordinal_alpha,
            )
        )

        self.ordinal_weight = float(
            ordinal_weight
        )

    def forward(
        self,
        logits,
        targets,
    ):
        ce = self.cross_entropy(
            logits,
            targets,
        )

        ordinal = self.ordinal(
            logits,
            targets,
        )

        total = (
            ce
            +
            self.ordinal_weight
            *
            ordinal
        )

        return (
            total,
            ce.detach(),
            ordinal.detach(),
        )


# ============================================================
# 23. Scheduler / AMP
# ============================================================

def build_scheduler(
    optimizer,
):
    def lr_lambda(
        epoch,
    ):
        if epoch < WARMUP_EPOCHS:
            return (
                float(
                    epoch + 1
                )
                /
                float(
                    WARMUP_EPOCHS
                )
            )

        progress = (
            float(
                epoch
                -
                WARMUP_EPOCHS
            )
            /
            float(
                max(
                    1,
                    EPOCHS
                    -
                    WARMUP_EPOCHS,
                )
            )
        )

        cosine = 0.5 * (
            1.0
            +
            math.cos(
                math.pi
                *
                progress
            )
        )

        return max(
            MIN_LR_RATIO,
            cosine,
        )

    return (
        torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda,
        )
    )


def create_scaler():
    try:
        return torch.amp.GradScaler(
            "cuda",

            enabled=(
                USE_MIXED_PRECISION
                and
                torch.cuda.is_available()
            ),
        )

    except Exception:
        return (
            torch.cuda.amp.GradScaler(
                enabled=(
                    USE_MIXED_PRECISION
                    and
                    torch.cuda.is_available()
                )
            )
        )


def amp_context(
    enabled,
):
    try:
        return torch.amp.autocast(
            device_type="cuda",
            enabled=enabled,
        )

    except Exception:
        return torch.cuda.amp.autocast(
            enabled=enabled
        )


# ============================================================
# 24. Class weights
# ============================================================

def calculate_class_weights(
    dataframe,
    device,
):
    counts = (
        dataframe[
            "label"
        ]
        .value_counts()
        .reindex(
            range(
                NUM_CLASSES
            ),

            fill_value=0,
        )
        .sort_index()
    )

    if (
        counts == 0
    ).any():
        raise RuntimeError(
            "Training fold missing a class."
        )

    weights = (
        counts.sum()
        /
        (
            NUM_CLASSES
            *
            counts
        )
    )

    return torch.tensor(
        weights.to_numpy(),
        dtype=torch.float32,
        device=device,
    )


# ============================================================
# 25. Train / evaluate
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
):
    model.train()

    total_loss = 0.0

    y_true = []
    y_pred = []

    use_amp = (
        USE_MIXED_PRECISION
        and
        device.type == "cuda"
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    number_of_batches = len(
        loader
    )

    for batch_index, batch in enumerate(
        loader
    ):
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        labels = batch["label"].to(
            device,
            non_blocking=True,
        )

        group_start = (
            batch_index
            //
            GRADIENT_ACCUMULATION_STEPS
        ) * GRADIENT_ACCUMULATION_STEPS

        group_size = min(
            GRADIENT_ACCUMULATION_STEPS,

            number_of_batches
            -
            group_start,
        )

        with amp_context(
            use_amp
        ):
            logits = model(
                images
            )

            raw_loss, _, _ = (
                criterion(
                    logits,
                    labels,
                )
            )

            loss = (
                raw_loss
                /
                group_size
            )

        if use_amp:
            scaler.scale(
                loss
            ).backward()

        else:
            loss.backward()

        should_step = (
            (
                batch_index + 1
            )
            %
            GRADIENT_ACCUMULATION_STEPS
            ==
            0
            or
            (
                batch_index + 1
            )
            ==
            number_of_batches
        )

        if should_step:
            if use_amp:
                scaler.unscale_(
                    optimizer
                )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),

                GRAD_CLIP_NORM,
            )

            if use_amp:
                scaler.step(
                    optimizer
                )

                scaler.update()

            else:
                optimizer.step()

            optimizer.zero_grad(
                set_to_none=True
            )

        total_loss += (
            float(
                raw_loss.item()
            )
            *
            images.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        y_true.extend(
            labels.detach()
            .cpu()
            .tolist()
        )

        y_pred.extend(
            predictions.detach()
            .cpu()
            .tolist()
        )

    metrics = compute_metrics(
        y_true,
        y_pred,
    )

    metrics["loss"] = (
        total_loss
        /
        len(
            loader.dataset
        )
    )

    return metrics


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
):
    model.eval()

    total_loss = 0.0

    y_true = []
    patient_truth = []
    patient_ids = []
    paths = []

    probability_parts = []

    use_amp = (
        USE_MIXED_PRECISION
        and
        device.type == "cuda"
    )

    for batch in loader:
        images = batch[
            "image"
        ].to(
            device,
            non_blocking=True,
        )

        labels = batch[
            "label"
        ].to(
            device,
            non_blocking=True,
        )

        with amp_context(
            use_amp
        ):
            logits = model(
                images
            )

            loss, _, _ = criterion(
                logits,
                labels,
            )

            probability_list = [
                torch.softmax(
                    logits,
                    dim=1,
                )
            ]

            if (
                USE_TTA
                and
                TTA_HORIZONTAL_FLIP
            ):
                flipped_logits = model(
                    torch.flip(
                        images,
                        dims=[3],
                    )
                )

                probability_list.append(
                    torch.softmax(
                        flipped_logits,
                        dim=1,
                    )
                )

            probabilities = (
                torch.stack(
                    probability_list,
                    dim=0,
                )
                .mean(
                    dim=0
                )
            )

        total_loss += (
            float(
                loss.item()
            )
            *
            images.size(0)
        )

        y_true.extend(
            labels.cpu().tolist()
        )

        patient_truth.extend(
            batch[
                "patient_y_true"
            ].cpu().tolist()
        )

        patient_ids.extend(
            batch[
                "patient_id"
            ].cpu().tolist()
        )

        paths.extend(
            batch["path"]
        )

        probability_parts.extend(
            probabilities.cpu()
            .numpy()
            .tolist()
        )

    image_df = pd.DataFrame({
        "path": paths,

        "patient_id": (
            patient_ids
        ),

        "y_true": (
            y_true
        ),

        "patient_y_true": (
            patient_truth
        ),
    })

    probability_df = pd.DataFrame(
        probability_parts,

        columns=(
            PROBABILITY_COLUMNS
        ),
    )

    image_df = pd.concat(
        [
            image_df,
            probability_df,
        ],

        axis=1,
    )

    image_df["y_pred"] = (
        image_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy()
        .argmax(
            axis=1
        )
        .astype(int)
    )

    image_metrics = (
        compute_metrics(
            image_df["y_true"],
            image_df["y_pred"],
        )
    )

    image_metrics["loss"] = (
        total_loss
        /
        len(
            loader.dataset
        )
    )

    patient_df = (
        aggregate_patient_probabilities(
            image_df
        )
    )

    patient_metrics = (
        compute_metrics(
            patient_df[
                "y_true"
            ],

            patient_df[
                "y_pred"
            ],
        )
    )

    return (
        image_metrics,
        patient_metrics,
        image_df,
        patient_df,
    )


# ============================================================
# 26. Checkpoint hierarchy
# ============================================================

def metric_max(
    value,
):
    if (
        value is None
        or
        not np.isfinite(
            value
        )
    ):
        return -np.inf

    return float(
        value
    )


def metric_min(
    value,
):
    if (
        value is None
        or
        not np.isfinite(
            value
        )
    ):
        return np.inf

    return float(
        value
    )


def checkpoint_is_better(
    patient_metrics,
    best_patient_metrics,
    image_metrics,
    best_image_metrics,
):
    if best_patient_metrics is None:
        return True

    candidate_tuple = (
        metric_max(
            patient_metrics[
                "qwk"
            ]
        ),

        -metric_min(
            patient_metrics[
                "mae"
            ]
        ),

        metric_max(
            patient_metrics[
                "macro_f1"
            ]
        ),

        metric_max(
            image_metrics[
                "qwk"
            ]
        ),
    )

    best_tuple = (
        metric_max(
            best_patient_metrics[
                "qwk"
            ]
        ),

        -metric_min(
            best_patient_metrics[
                "mae"
            ]
        ),

        metric_max(
            best_patient_metrics[
                "macro_f1"
            ]
        ),

        metric_max(
            best_image_metrics[
                "qwk"
            ]
        ),
    )

    return (
        candidate_tuple
        >
        best_tuple
    )


# ============================================================
# 27. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "\nDevice:",
    device,
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        ),
    )


# ============================================================
# 28. Run/resume one fold
# ============================================================

def run_fold(
    backbone_name,
    backbone_config,
    training_seed,
    fold,
    seed_directory,
):
    fold_seed = (
        int(training_seed)
        +
        int(fold)
    )

    fold_directory = (
        seed_directory
        /
        f"fold_{fold}"
    )

    fold_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    fold_config = {
        "protocol_version": (
            PROTOCOL_VERSION
        ),

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),

        "backbone_name": (
            backbone_name
        ),

        "backbone_config": (
            backbone_config
        ),

        "training_seed": (
            training_seed
        ),

        "fold": (
            fold
        ),

        "fold_seed": (
            fold_seed
        ),
    }

    fold_fingerprint = (
        dictionary_sha256(
            fold_config
        )
    )

    train_df = (
        trainval_df[
            trainval_df[
                "cv_fold"
            ]
            !=
            fold
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    validation_df = (
        trainval_df[
            trainval_df[
                "cv_fold"
            ]
            ==
            fold
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    completed_path = (
        fold_directory
        / "fold_result.json"
    )

    image_prediction_path = (
        fold_directory
        / "validation_image_predictions.csv"
    )

    patient_prediction_path = (
        fold_directory
        / "validation_patient_predictions.csv"
    )

    best_checkpoint_path = (
        fold_directory
        / "best_checkpoint.pt"
    )

    last_checkpoint_path = (
        fold_directory
        / "last_epoch_checkpoint.pt"
    )

    history_path = (
        fold_directory
        / "history.csv"
    )

    if (
        RESUME_COMPLETED_FOLDS
        and
        completed_path.exists()
        and
        image_prediction_path.exists()
        and
        patient_prediction_path.exists()
        and
        best_checkpoint_path.exists()
    ):
        with open(
            completed_path,
            "r",
            encoding="utf-8",
        ) as file:
            completed_result = (
                json.load(file)
            )

        if (
            completed_result[
                "fold_fingerprint"
            ]
            !=
            fold_fingerprint
        ):
            raise RuntimeError(
                "Completed fold fingerprint mismatch."
            )

        print(
            "\nLoaded completed fold:",
            backbone_name,
            "| seed",
            training_seed,
            "| fold",
            fold,
        )

        return {
            "fold_result": (
                completed_result
            ),

            "image_predictions": (
                pd.read_csv(
                    image_prediction_path
                )
            ),

            "patient_predictions": (
                pd.read_csv(
                    patient_prediction_path
                )
            ),
        }

    print(
        "\n"
        +
        "=" * 110
    )

    print(
        f"{backbone_name} | "
        f"seed={training_seed} | "
        f"fold={fold + 1}/{NUM_FOLDS}"
    )

    print(
        "=" * 110
    )

    print(
        "Train images:",
        len(train_df),
    )

    print(
        "Validation images:",
        len(validation_df),
    )

    train_dataset = LIMUCDataset(
        train_df,
        transform=(
            build_train_transform()
        ),
    )

    validation_dataset = LIMUCDataset(
        validation_df,
        transform=(
            build_eval_transform()
        ),
    )

    training_generator = (
        torch.Generator()
    )

    validation_generator = (
        torch.Generator()
    )

    training_generator.manual_seed(
        fold_seed
    )

    validation_generator.manual_seed(
        fold_seed
        +
        100000
    )

    train_loader = DataLoader(
        train_dataset,

        batch_size=(
            TRAIN_BATCH_SIZE
        ),

        shuffle=True,

        num_workers=(
            RUNTIME_NUM_WORKERS
        ),

        pin_memory=(
            device.type
            ==
            "cuda"
        ),

        generator=(
            training_generator
        ),

        worker_init_fn=(
            seed_worker
        ),

        persistent_workers=False,
    )

    validation_loader = DataLoader(
        validation_dataset,

        batch_size=(
            VALIDATION_BATCH_SIZE
        ),

        shuffle=False,

        num_workers=(
            RUNTIME_NUM_WORKERS
        ),

        pin_memory=(
            device.type
            ==
            "cuda"
        ),

        generator=(
            validation_generator
        ),

        worker_init_fn=(
            seed_worker
        ),

        persistent_workers=False,
    )

    resume_payload = None

    if (
        RESUME_INTERRUPTED_FOLDS
        and
        last_checkpoint_path.exists()
    ):
        resume_payload = safe_torch_load(
            last_checkpoint_path,
            map_location="cpu",
        )

        if (
            resume_payload[
                "fold_fingerprint"
            ]
            !=
            fold_fingerprint
        ):
            raise RuntimeError(
                "Resume fingerprint mismatch."
            )

        print(
            "\nFound resumable checkpoint:",
            last_checkpoint_path,
        )

    seed_everything(
        fold_seed
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = timm.create_model(
        backbone_config[
            "model_name"
        ],

        pretrained=(
            resume_payload
            is
            None
        ),

        num_classes=(
            NUM_CLASSES
        ),

        drop_rate=(
            DROP_RATE
        ),

        drop_path_rate=(
            DROP_PATH_RATE
        ),
    ).to(
        device
    )

    parameter_count = int(
        sum(
            parameter.numel()
            for parameter in (
                model.parameters()
            )
        )
    )

    class_weights = (
        calculate_class_weights(
            train_df,
            device,
        )
    )

    criterion = CombinedLoss(
        ce_weight=(
            class_weights
            if
            USE_CLASS_WEIGHTS_IN_CE
            else
            None
        ),

        ordinal_alpha=(
            backbone_config[
                "ordinal_alpha"
            ]
        ),

        ordinal_weight=(
            backbone_config[
                "ordinal_loss_weight"
            ]
        ),
    ).to(
        device
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),

        lr=(
            LEARNING_RATE
        ),

        weight_decay=(
            WEIGHT_DECAY
        ),
    )

    scheduler = build_scheduler(
        optimizer
    )

    scaler = create_scaler()

    start_epoch = 1

    best_epoch = -1

    best_patient_metrics = None

    best_image_metrics = None

    patience_counter = 0

    history = []

    if resume_payload is not None:
        model.load_state_dict(
            resume_payload[
                "model_state_dict"
            ]
        )

        optimizer.load_state_dict(
            resume_payload[
                "optimizer_state_dict"
            ]
        )

        optimizer_to_device(
            optimizer,
            device,
        )

        scheduler.load_state_dict(
            resume_payload[
                "scheduler_state_dict"
            ]
        )

        scaler.load_state_dict(
            resume_payload[
                "scaler_state_dict"
            ]
        )

        completed_epoch = int(
            resume_payload[
                "completed_epoch"
            ]
        )

        start_epoch = (
            completed_epoch
            +
            1
        )

        best_epoch = int(
            resume_payload[
                "best_epoch"
            ]
        )

        best_patient_metrics = (
            resume_payload[
                "best_patient_metrics"
            ]
        )

        best_image_metrics = (
            resume_payload[
                "best_image_metrics"
            ]
        )

        patience_counter = int(
            resume_payload[
                "patience_counter"
            ]
        )

        history = (
            resume_payload[
                "history"
            ]
        )

        training_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "training_generator_state"
                ],

                "train generator",
            )
        )

        validation_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "validation_generator_state"
                ],

                "validation generator",
            )
        )

        restore_rng_state(
            resume_payload[
                "rng_state"
            ]
        )

        print(
            "Resuming from epoch:",
            completed_epoch,
        )

        print(
            "Next epoch:",
            start_epoch,
        )

    for epoch in range(
        start_epoch,
        EPOCHS + 1,
    ):
        if (
            patience_counter
            >=
            EARLY_STOP_PATIENCE
        ):
            break

        train_metrics = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            device,
        )

        (
            validation_image_metrics,
            validation_patient_metrics,
            _,
            _,
        ) = evaluate(
            model,
            validation_loader,
            criterion,
            device,
        )

        current_lr = float(
            optimizer.param_groups[
                0
            ][
                "lr"
            ]
        )

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"QWK={validation_patient_metrics['qwk']:.4f} | "
            f"MAE={validation_patient_metrics['mae']:.4f} | "
            f"F1={validation_patient_metrics['macro_f1']:.4f} | "
            f"imageQWK={validation_image_metrics['qwk']:.4f}"
        )

        improved = checkpoint_is_better(
            validation_patient_metrics,

            best_patient_metrics,

            validation_image_metrics,

            best_image_metrics,
        )

        if improved:
            best_epoch = epoch

            best_patient_metrics = (
                copy.deepcopy(
                    validation_patient_metrics
                )
            )

            best_image_metrics = (
                copy.deepcopy(
                    validation_image_metrics
                )
            )

            patience_counter = 0

            atomic_torch_save(
                {
                    "fold_fingerprint": (
                        fold_fingerprint
                    ),

                    "backbone_name": (
                        backbone_name
                    ),

                    "training_seed": (
                        training_seed
                    ),

                    "fold": (
                        fold
                    ),

                    "best_epoch": (
                        best_epoch
                    ),

                    "model_state_dict": {
                        key: value.detach().cpu()
                        for key, value in (
                            model.state_dict().items()
                        )
                    },

                    "best_patient_metrics": (
                        best_patient_metrics
                    ),

                    "best_image_metrics": (
                        best_image_metrics
                    ),
                },

                best_checkpoint_path,
            )

        else:
            patience_counter += 1

        history.append({
            "epoch": epoch,

            "lr": current_lr,

            "training_loss": (
                train_metrics[
                    "loss"
                ]
            ),

            "validation_image_qwk": (
                validation_image_metrics[
                    "qwk"
                ]
            ),

            "validation_patient_qwk": (
                validation_patient_metrics[
                    "qwk"
                ]
            ),

            "validation_patient_mae": (
                validation_patient_metrics[
                    "mae"
                ]
            ),

            "validation_patient_macro_f1": (
                validation_patient_metrics[
                    "macro_f1"
                ]
            ),

            "validation_patient_accuracy": (
                validation_patient_metrics[
                    "accuracy"
                ]
            ),
        })

        atomic_csv_save(
            pd.DataFrame(
                history
            ),

            history_path,

            index=False,
        )

        scheduler.step()

        atomic_torch_save(
            {
                "fold_fingerprint": (
                    fold_fingerprint
                ),

                "completed_epoch": (
                    epoch
                ),

                "best_epoch": (
                    best_epoch
                ),

                "best_patient_metrics": (
                    best_patient_metrics
                ),

                "best_image_metrics": (
                    best_image_metrics
                ),

                "patience_counter": (
                    patience_counter
                ),

                "history": (
                    history
                ),

                "model_state_dict": (
                    model.state_dict()
                ),

                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),

                "scheduler_state_dict": (
                    scheduler.state_dict()
                ),

                "scaler_state_dict": (
                    scaler.state_dict()
                ),

                "rng_state": (
                    capture_rng_state()
                ),

                "training_generator_state": (
                    training_generator
                    .get_state()
                    .cpu()
                ),

                "validation_generator_state": (
                    validation_generator
                    .get_state()
                    .cpu()
                ),
            },

            last_checkpoint_path,
        )

    if not best_checkpoint_path.exists():
        raise RuntimeError(
            "Best checkpoint not found."
        )

    best_checkpoint = safe_torch_load(
        best_checkpoint_path,
        map_location="cpu",
    )

    model.load_state_dict(
        best_checkpoint[
            "model_state_dict"
        ]
    )

    (
        image_metrics,
        patient_metrics,
        image_df,
        patient_df,
    ) = evaluate(
        model,
        validation_loader,
        criterion,
        device,
    )

    image_df["fold"] = fold

    image_df["training_seed"] = (
        training_seed
    )

    patient_df["fold"] = fold

    patient_df["training_seed"] = (
        training_seed
    )

    atomic_csv_save(
        image_df,

        image_prediction_path,

        index=False,
    )

    atomic_csv_save(
        patient_df,

        patient_prediction_path,

        index=False,
    )

    fold_result = {
        "completed_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "fold_fingerprint": (
            fold_fingerprint
        ),

        "backbone_name": (
            backbone_name
        ),

        "model_name": (
            backbone_config[
                "model_name"
            ]
        ),

        "representative_configuration": (
            backbone_config[
                "representative_configuration"
            ]
        ),

        "training_seed": (
            training_seed
        ),

        "fold": fold,

        "best_epoch": (
            best_checkpoint[
                "best_epoch"
            ]
        ),

        "parameter_count": (
            parameter_count
        ),

        "image_metrics": (
            image_metrics
        ),

        "patient_metrics": (
            patient_metrics
        ),

        "official_test_loaded": False,

        "official_test_evaluated": False,
    }

    atomic_json_save(
        fold_result,

        completed_path,
    )

    if last_checkpoint_path.exists():
        last_checkpoint_path.unlink()

    del model
    del optimizer
    del scheduler
    del criterion
    del scaler
    del best_checkpoint

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "fold_result": (
            fold_result
        ),

        "image_predictions": (
            image_df
        ),

        "patient_predictions": (
            patient_df
        ),
    }


# ============================================================
# 29. Bootstrap function
# ============================================================

def stratified_bootstrap_metrics(
    patient_df,
    repetitions,
    seed,
):
    patient_df = (
        patient_df
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )

    y_true = (
        patient_df[
            "y_true"
        ]
        .to_numpy(
            dtype=int
        )
    )

    y_pred = (
        patient_df[
            "y_pred"
        ]
        .to_numpy(
            dtype=int
        )
    )

    class_indices = {
        class_index: np.flatnonzero(
            y_true
            ==
            class_index
        )

        for class_index in range(
            NUM_CLASSES
        )
    }

    rng = np.random.default_rng(
        seed
    )

    rows = []

    for bootstrap_index in range(
        repetitions
    ):
        sampled_indices = np.concatenate([
            rng.choice(
                class_indices[
                    class_index
                ],

                size=len(
                    class_indices[
                        class_index
                    ]
                ),

                replace=True,
            )

            for class_index in range(
                NUM_CLASSES
            )
        ])

        metrics = compute_metrics(
            y_true[
                sampled_indices
            ],

            y_pred[
                sampled_indices
            ],
        )

        rows.append({
            "bootstrap_index": (
                bootstrap_index
            ),

            **metrics,
        })

    bootstrap_df = pd.DataFrame(
        rows
    )

    ci_rows = []

    for metric_name in [
        "qwk",
        "mae",
        "macro_f1",
        "accuracy",
    ]:
        values = (
            bootstrap_df[
                metric_name
            ]
            .dropna()
            .to_numpy()
        )

        ci_rows.append({
            "metric": (
                metric_name
            ),

            "lower_95": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

            "upper_95": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),
        })

    return (
        pd.DataFrame(
            ci_rows
        ),

        bootstrap_df,
    )


# ============================================================
# 30. Run one complete backbone-seed experiment
# ============================================================

def run_backbone_seed(
    backbone_name,
    training_seed,
):
    backbone_config = (
        BACKBONE_CONFIGS[
            backbone_name
        ]
    )

    seed_directory = (
        OUTPUT_ROOT
        /
        backbone_name
        /
        f"seed_{training_seed}"
    )

    seed_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    seed_result_path = (
        seed_directory
        / "seed_result.json"
    )

    oof_image_path = (
        seed_directory
        / "oof_image_predictions.csv"
    )

    oof_patient_path = (
        seed_directory
        / "oof_patient_predictions.csv"
    )

    bootstrap_ci_path = (
        seed_directory
        / "oof_patient_bootstrap_ci.csv"
    )

    bootstrap_samples_path = (
        seed_directory
        / "oof_patient_bootstrap_samples.csv"
    )

    seed_configuration = {
        "protocol_version": (
            PROTOCOL_VERSION
        ),

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),

        "backbone_name": (
            backbone_name
        ),

        "backbone_config": (
            backbone_config
        ),

        "training_seed": (
            training_seed
        ),
    }

    seed_fingerprint = (
        dictionary_sha256(
            seed_configuration
        )
    )

    if (
        SKIP_COMPLETED_SEED_EXPERIMENTS
        and
        seed_result_path.exists()
        and
        oof_image_path.exists()
        and
        oof_patient_path.exists()
        and
        bootstrap_ci_path.exists()
    ):
        with open(
            seed_result_path,
            "r",
            encoding="utf-8",
        ) as file:
            seed_result = (
                json.load(file)
            )

        if (
            seed_result[
                "seed_fingerprint"
            ]
            !=
            seed_fingerprint
        ):
            raise RuntimeError(
                "Completed seed fingerprint mismatch."
            )

        print(
            "\nLoaded completed seed:",
            backbone_name,
            training_seed,
        )

        return {
            "result_row": (
                seed_result
            ),

            "oof_image_df": (
                pd.read_csv(
                    oof_image_path
                )
            ),

            "oof_patient_df": (
                pd.read_csv(
                    oof_patient_path
                )
            ),
        }

    fold_image_parts = []

    fold_rows = []

    for fold in range(
        NUM_FOLDS
    ):
        fold_output = run_fold(
            backbone_name,
            backbone_config,
            training_seed,
            fold,
            seed_directory,
        )

        fold_image_parts.append(
            fold_output[
                "image_predictions"
            ]
        )

        fold_result = (
            fold_output[
                "fold_result"
            ]
        )

        fold_rows.append({
            "backbone_name": (
                backbone_name
            ),

            "training_seed": (
                training_seed
            ),

            "fold": fold,

            "best_epoch": (
                fold_result[
                    "best_epoch"
                ]
            ),

            "patient_qwk": (
                fold_result[
                    "patient_metrics"
                ][
                    "qwk"
                ]
            ),

            "patient_mae": (
                fold_result[
                    "patient_metrics"
                ][
                    "mae"
                ]
            ),

            "patient_macro_f1": (
                fold_result[
                    "patient_metrics"
                ][
                    "macro_f1"
                ]
            ),

            "patient_accuracy": (
                fold_result[
                    "patient_metrics"
                ][
                    "accuracy"
                ]
            ),

            "image_qwk": (
                fold_result[
                    "image_metrics"
                ][
                    "qwk"
                ]
            ),
        })

    oof_image_df = (
        pd.concat(
            fold_image_parts,
            ignore_index=True,
        )
        .sort_values(
            "path"
        )
        .reset_index(
            drop=True
        )
    )

    if len(oof_image_df) != 9590:
        raise RuntimeError(
            "OOF image count is not 9590."
        )

    if (
        oof_image_df[
            "path"
        ].duplicated().any()
    ):
        raise RuntimeError(
            "Duplicate OOF image predictions."
        )

    if (
        oof_image_df
        .groupby(
            "patient_id"
        )[
            "fold"
        ]
        .nunique()
        .max()
        !=
        1
    ):
        raise RuntimeError(
            "Patient appears in multiple OOF folds."
        )

    oof_patient_df = (
        aggregate_patient_probabilities(
            oof_image_df
        )
    )

    if len(oof_patient_df) != 478:
        raise RuntimeError(
            "OOF patient count is not 478."
        )

    image_metrics = (
        compute_metrics(
            oof_image_df[
                "y_true"
            ],

            oof_image_df[
                "y_pred"
            ],
        )
    )

    patient_metrics = (
        compute_metrics(
            oof_patient_df[
                "y_true"
            ],

            oof_patient_df[
                "y_pred"
            ],
        )
    )

    (
        bootstrap_ci_df,
        bootstrap_samples_df,
    ) = stratified_bootstrap_metrics(
        oof_patient_df,

        BOOTSTRAP_REPETITIONS,

        BOOTSTRAP_SEED
        +
        BACKBONES_TO_RUN.index(
            backbone_name
        )
        *
        10000
        +
        CONFIRMATION_SEEDS.index(
            training_seed
        )
        *
        1000,
    )

    atomic_csv_save(
        oof_image_df,

        oof_image_path,

        index=False,
    )

    atomic_csv_save(
        oof_patient_df,

        oof_patient_path,

        index=False,
    )

    atomic_csv_save(
        pd.DataFrame(
            fold_rows
        ),

        seed_directory
        / "fold_summary.csv",

        index=False,
    )

    atomic_csv_save(
        bootstrap_ci_df,

        bootstrap_ci_path,

        index=False,
    )

    atomic_csv_save(
        bootstrap_samples_df,

        bootstrap_samples_path,

        index=False,
    )

    atomic_csv_save(
        make_confusion_df(
            oof_patient_df[
                "y_true"
            ],

            oof_patient_df[
                "y_pred"
            ],
        ),

        seed_directory
        / "oof_patient_confusion_matrix.csv",

        index=True,
    )

    ci_lookup = (
        bootstrap_ci_df
        .set_index(
            "metric"
        )
    )

    result_row = {
        "backbone_name": (
            backbone_name
        ),

        "model_name": (
            backbone_config[
                "model_name"
            ]
        ),

        "representative_configuration": (
            backbone_config[
                "representative_configuration"
            ]
        ),

        "ordinal_alpha": (
            backbone_config[
                "ordinal_alpha"
            ]
        ),

        "ordinal_loss_weight": (
            backbone_config[
                "ordinal_loss_weight"
            ]
        ),

        "training_seed": (
            training_seed
        ),

        "seed_fingerprint": (
            seed_fingerprint
        ),

        "oof_patient_qwk": (
            patient_metrics[
                "qwk"
            ]
        ),

        "oof_patient_qwk_ci_low": float(
            ci_lookup.loc[
                "qwk",
                "lower_95",
            ]
        ),

        "oof_patient_qwk_ci_high": float(
            ci_lookup.loc[
                "qwk",
                "upper_95",
            ]
        ),

        "oof_patient_mae": (
            patient_metrics[
                "mae"
            ]
        ),

        "oof_patient_macro_f1": (
            patient_metrics[
                "macro_f1"
            ]
        ),

        "oof_patient_accuracy": (
            patient_metrics[
                "accuracy"
            ]
        ),

        "oof_image_qwk": (
            image_metrics[
                "qwk"
            ]
        ),

        "oof_image_mae": (
            image_metrics[
                "mae"
            ]
        ),

        "oof_image_macro_f1": (
            image_metrics[
                "macro_f1"
            ]
        ),

        "oof_image_accuracy": (
            image_metrics[
                "accuracy"
            ]
        ),

        "official_test_loaded": False,

        "official_test_evaluated": False,
    }

    atomic_json_save(
        result_row,

        seed_result_path,
    )

    print(
        "\nCompleted:",
        backbone_name,
        "| seed",
        training_seed,
    )

    print(
        "Patient metrics:",
        patient_metrics,
    )

    return {
        "result_row": (
            result_row
        ),

        "oof_image_df": (
            oof_image_df
        ),

        "oof_patient_df": (
            oof_patient_df
        ),
    }


# ============================================================
# 31. Execute 2 backbones x 3 seeds
# ============================================================

all_outputs = {}

result_rows = []

patient_prediction_lookup = {}

image_prediction_lookup = {}

for backbone_name in (
    BACKBONES_TO_RUN
):
    all_outputs[
        backbone_name
    ] = {}

    patient_prediction_lookup[
        backbone_name
    ] = {}

    image_prediction_lookup[
        backbone_name
    ] = {}

    for training_seed in (
        CONFIRMATION_SEEDS
    ):
        output = run_backbone_seed(
            backbone_name,
            training_seed,
        )

        all_outputs[
            backbone_name
        ][
            training_seed
        ] = output

        patient_prediction_lookup[
            backbone_name
        ][
            training_seed
        ] = (
            output[
                "oof_patient_df"
            ]
        )

        image_prediction_lookup[
            backbone_name
        ][
            training_seed
        ] = (
            output[
                "oof_image_df"
            ]
        )

        result_rows.append(
            output[
                "result_row"
            ]
        )


# ============================================================
# 32. Three-seed summary
# ============================================================

per_seed_df = pd.DataFrame(
    result_rows
)

per_seed_path = (
    OUTPUT_ROOT
    / "backbone_results_by_seed.csv"
)

atomic_csv_save(
    per_seed_df,
    per_seed_path,
    index=False,
)

summary_df = (
    per_seed_df
    .groupby(
        "backbone_name",
        as_index=False,
    )
    .agg(
        n_seeds=(
            "training_seed",
            "nunique",
        ),

        patient_qwk_mean=(
            "oof_patient_qwk",
            "mean",
        ),

        patient_qwk_std=(
            "oof_patient_qwk",
            "std",
        ),

        patient_mae_mean=(
            "oof_patient_mae",
            "mean",
        ),

        patient_mae_std=(
            "oof_patient_mae",
            "std",
        ),

        patient_macro_f1_mean=(
            "oof_patient_macro_f1",
            "mean",
        ),

        patient_macro_f1_std=(
            "oof_patient_macro_f1",
            "std",
        ),

        patient_accuracy_mean=(
            "oof_patient_accuracy",
            "mean",
        ),

        patient_accuracy_std=(
            "oof_patient_accuracy",
            "std",
        ),

        image_qwk_mean=(
            "oof_image_qwk",
            "mean",
        ),

        image_qwk_std=(
            "oof_image_qwk",
            "std",
        ),

        image_mae_mean=(
            "oof_image_mae",
            "mean",
        ),

        image_mae_std=(
            "oof_image_mae",
            "std",
        ),

        image_macro_f1_mean=(
            "oof_image_macro_f1",
            "mean",
        ),

        image_macro_f1_std=(
            "oof_image_macro_f1",
            "std",
        ),

        image_accuracy_mean=(
            "oof_image_accuracy",
            "mean",
        ),

        image_accuracy_std=(
            "oof_image_accuracy",
            "std",
        ),
    )
)

ranked_df = (
    summary_df
    .sort_values(
        by=[
            "patient_qwk_mean",
            "patient_mae_mean",
            "patient_macro_f1_mean",
            "patient_accuracy_mean",
            "image_qwk_mean",
        ],

        ascending=[
            False,
            True,
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

ranked_df.insert(
    0,
    "development_rank",
    np.arange(
        1,
        len(
            ranked_df
        ) + 1,
    ),
)

summary_path = (
    OUTPUT_ROOT
    / "backbone_multiseed_summary.csv"
)

ranking_path = (
    OUTPUT_ROOT
    / "backbone_multiseed_ranking.csv"
)

atomic_csv_save(
    summary_df,
    summary_path,
    index=False,
)

atomic_csv_save(
    ranked_df,
    ranking_path,
    index=False,
)


# ============================================================
# 33. Paired bootstrap: mean metrics across seeds
# ============================================================

BACKBONE_A = str(
    ranked_df.iloc[
        0
    ][
        "backbone_name"
    ]
)

BACKBONE_B = str(
    ranked_df.iloc[
        1
    ][
        "backbone_name"
    ]
)

base_patient_df = (
    patient_prediction_lookup[
        BACKBONE_A
    ][
        CONFIRMATION_SEEDS[0]
    ]
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)

patient_ids = (
    base_patient_df[
        "patient_id"
    ]
    .to_numpy(
        dtype=int
    )
)

patient_truth = (
    base_patient_df[
        "y_true"
    ]
    .to_numpy(
        dtype=int
    )
)

prediction_arrays = {}

for backbone_name in (
    BACKBONES_TO_RUN
):
    for training_seed in (
        CONFIRMATION_SEEDS
    ):
        patient_df = (
            patient_prediction_lookup[
                backbone_name
            ][
                training_seed
            ]
            .sort_values(
                "patient_id"
            )
            .reset_index(
                drop=True
            )
        )

        if not np.array_equal(
            patient_df[
                "patient_id"
            ].to_numpy(
                dtype=int
            ),

            patient_ids,
        ):
            raise RuntimeError(
                "Patient alignment failure."
            )

        if not np.array_equal(
            patient_df[
                "y_true"
            ].to_numpy(
                dtype=int
            ),

            patient_truth,
        ):
            raise RuntimeError(
                "Reference alignment failure."
            )

        prediction_arrays[
            (
                backbone_name,
                training_seed,
            )
        ] = (
            patient_df[
                "y_pred"
            ]
            .to_numpy(
                dtype=int
            )
        )

class_indices = {
    class_index: np.flatnonzero(
        patient_truth
        ==
        class_index
    )

    for class_index in range(
        NUM_CLASSES
    )
}

rng = np.random.default_rng(
    BOOTSTRAP_SEED
    +
    500000
)

paired_rows = []

for bootstrap_index in range(
    BOOTSTRAP_REPETITIONS
):
    sampled_indices = np.concatenate([
        rng.choice(
            class_indices[
                class_index
            ],

            size=len(
                class_indices[
                    class_index
                ]
            ),

            replace=True,
        )

        for class_index in range(
            NUM_CLASSES
        )
    ])

    sampled_true = (
        patient_truth[
            sampled_indices
        ]
    )

    mean_metrics = {}

    for backbone_name in [
        BACKBONE_A,
        BACKBONE_B,
    ]:
        metric_values = {
            "qwk": [],
            "mae": [],
            "macro_f1": [],
            "accuracy": [],
        }

        for training_seed in (
            CONFIRMATION_SEEDS
        ):
            metrics = compute_metrics(
                sampled_true,

                prediction_arrays[
                    (
                        backbone_name,
                        training_seed,
                    )
                ][
                    sampled_indices
                ],
            )

            for metric_name in (
                metric_values
            ):
                metric_values[
                    metric_name
                ].append(
                    metrics[
                        metric_name
                    ]
                )

        mean_metrics[
            backbone_name
        ] = {
            metric_name: float(
                np.mean(
                    values
                )
            )

            for metric_name, values in (
                metric_values.items()
            )
        }

    a = mean_metrics[
        BACKBONE_A
    ]

    b = mean_metrics[
        BACKBONE_B
    ]

    paired_rows.append({
        "bootstrap_index": (
            bootstrap_index
        ),

        "delta_qwk": (
            a["qwk"]
            -
            b["qwk"]
        ),

        "delta_mae": (
            a["mae"]
            -
            b["mae"]
        ),

        "delta_macro_f1": (
            a["macro_f1"]
            -
            b["macro_f1"]
        ),

        "delta_accuracy": (
            a["accuracy"]
            -
            b["accuracy"]
        ),
    })

    if (
        bootstrap_index + 1
    ) % 500 == 0:
        print(
            "Completed paired bootstrap:",
            bootstrap_index + 1,
            "/",
            BOOTSTRAP_REPETITIONS,
        )

paired_samples_df = pd.DataFrame(
    paired_rows
)

metric_columns = {
    "qwk": "delta_qwk",

    "mae": "delta_mae",

    "macro_f1": (
        "delta_macro_f1"
    ),

    "accuracy": (
        "delta_accuracy"
    ),
}

a_summary = (
    ranked_df[
        ranked_df[
            "backbone_name"
        ]
        ==
        BACKBONE_A
    ]
    .iloc[0]
)

b_summary = (
    ranked_df[
        ranked_df[
            "backbone_name"
        ]
        ==
        BACKBONE_B
    ]
    .iloc[0]
)

point_metric_columns = {
    "qwk": (
        "patient_qwk_mean"
    ),

    "mae": (
        "patient_mae_mean"
    ),

    "macro_f1": (
        "patient_macro_f1_mean"
    ),

    "accuracy": (
        "patient_accuracy_mean"
    ),
}

paired_summary_rows = []

for metric_name, column in (
    metric_columns.items()
):
    values = (
        paired_samples_df[
            column
        ]
        .dropna()
        .to_numpy()
    )

    point_difference = (
        float(
            a_summary[
                point_metric_columns[
                    metric_name
                ]
            ]
        )
        -
        float(
            b_summary[
                point_metric_columns[
                    metric_name
                ]
            ]
        )
    )

    probability_nonpositive = float(
        np.mean(
            values <= 0
        )
    )

    probability_nonnegative = float(
        np.mean(
            values >= 0
        )
    )

    tail_proportion = min(
        1.0,

        2.0
        *
        min(
            probability_nonpositive,
            probability_nonnegative,
        ),
    )

    paired_summary_rows.append({
        "metric": (
            metric_name
        ),

        "backbone_a": (
            BACKBONE_A
        ),

        "backbone_b": (
            BACKBONE_B
        ),

        "difference_definition": (
            f"{BACKBONE_A} - {BACKBONE_B}"
        ),

        "point_difference": (
            point_difference
        ),

        "difference_ci_low": float(
            np.percentile(
                values,
                2.5,
            )
        ),

        "difference_ci_high": float(
            np.percentile(
                values,
                97.5,
            )
        ),

        "bootstrap_two_sided_tail_proportion": (
            tail_proportion
        ),

        "direction_note": (
            "Negative favors backbone A"
            if
            metric_name
            ==
            "mae"
            else
            "Positive favors backbone A"
        ),
    })

paired_summary_df = pd.DataFrame(
    paired_summary_rows
)

paired_samples_path = (
    OUTPUT_ROOT
    / "backbone_paired_bootstrap_samples.csv"
)

paired_summary_path = (
    OUTPUT_ROOT
    / "backbone_paired_bootstrap_summary.csv"
)

atomic_csv_save(
    paired_samples_df,
    paired_samples_path,
    index=False,
)

atomic_csv_save(
    paired_summary_df,
    paired_summary_path,
    index=False,
)


# ============================================================
# 34. Seed-ensemble OOF predictions — supplementary
# ============================================================

def make_seed_ensemble(
    backbone_name,
):
    seed_frames = []

    for seed in (
        CONFIRMATION_SEEDS
    ):
        seed_frames.append(
            image_prediction_lookup[
                backbone_name
            ][
                seed
            ]
            .sort_values(
                "path"
            )
            .reset_index(
                drop=True
            )
        )

    base = seed_frames[
        0
    ][
        [
            "path",
            "patient_id",
            "y_true",
            "patient_y_true",
            "fold",
        ]
    ].copy()

    probabilities = []

    for frame in seed_frames:
        if not np.array_equal(
            frame[
                "path"
            ].to_numpy(),

            base[
                "path"
            ].to_numpy(),
        ):
            raise RuntimeError(
                "Image alignment failure "
                "during seed ensemble."
            )

        probabilities.append(
            frame[
                PROBABILITY_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )

    mean_probability = (
        np.mean(
            np.stack(
                probabilities,
                axis=0,
            ),

            axis=0,
        )
    )

    ensemble_df = base.copy()

    for class_index, column in enumerate(
        PROBABILITY_COLUMNS
    ):
        ensemble_df[
            column
        ] = (
            mean_probability[
                :,
                class_index,
            ]
        )

    ensemble_df[
        "y_pred"
    ] = (
        mean_probability
        .argmax(
            axis=1
        )
        .astype(int)
    )

    patient_df = (
        aggregate_patient_probabilities(
            ensemble_df
        )
    )

    return (
        ensemble_df,
        patient_df,
    )


seed_ensemble_rows = []

seed_ensemble_patient_lookup = {}

for backbone_name in (
    BACKBONES_TO_RUN
):
    (
        ensemble_image_df,
        ensemble_patient_df,
    ) = make_seed_ensemble(
        backbone_name
    )

    image_metrics = (
        compute_metrics(
            ensemble_image_df[
                "y_true"
            ],

            ensemble_image_df[
                "y_pred"
            ],
        )
    )

    patient_metrics = (
        compute_metrics(
            ensemble_patient_df[
                "y_true"
            ],

            ensemble_patient_df[
                "y_pred"
            ],
        )
    )

    (
        ci_df,
        bootstrap_df,
    ) = stratified_bootstrap_metrics(
        ensemble_patient_df,

        BOOTSTRAP_REPETITIONS,

        BOOTSTRAP_SEED
        +
        600000
        +
        BACKBONES_TO_RUN.index(
            backbone_name
        )
        *
        1000,
    )

    ensemble_directory = (
        OUTPUT_ROOT
        /
        backbone_name
        /
        "seed_ensemble"
    )

    ensemble_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    atomic_csv_save(
        ensemble_image_df,

        ensemble_directory
        / "oof_image_predictions_seed_ensemble.csv",

        index=False,
    )

    atomic_csv_save(
        ensemble_patient_df,

        ensemble_directory
        / "oof_patient_predictions_seed_ensemble.csv",

        index=False,
    )

    atomic_csv_save(
        ci_df,

        ensemble_directory
        / "oof_patient_bootstrap_ci.csv",

        index=False,
    )

    ci_lookup = (
        ci_df
        .set_index(
            "metric"
        )
    )

    seed_ensemble_rows.append({
        "backbone_name": (
            backbone_name
        ),

        "patient_qwk": (
            patient_metrics[
                "qwk"
            ]
        ),

        "patient_qwk_ci_low": float(
            ci_lookup.loc[
                "qwk",
                "lower_95",
            ]
        ),

        "patient_qwk_ci_high": float(
            ci_lookup.loc[
                "qwk",
                "upper_95",
            ]
        ),

        "patient_mae": (
            patient_metrics[
                "mae"
            ]
        ),

        "patient_macro_f1": (
            patient_metrics[
                "macro_f1"
            ]
        ),

        "patient_accuracy": (
            patient_metrics[
                "accuracy"
            ]
        ),

        "image_qwk": (
            image_metrics[
                "qwk"
            ]
        ),

        "image_mae": (
            image_metrics[
                "mae"
            ]
        ),
    })

    seed_ensemble_patient_lookup[
        backbone_name
    ] = (
        ensemble_patient_df
    )

seed_ensemble_df = pd.DataFrame(
    seed_ensemble_rows
)

seed_ensemble_path = (
    OUTPUT_ROOT
    / "backbone_seed_ensemble_results.csv"
)

atomic_csv_save(
    seed_ensemble_df,
    seed_ensemble_path,
    index=False,
)


# ============================================================
# 35. Lock representative backbone
# ============================================================

qwk_pair_row = (
    paired_summary_df[
        paired_summary_df[
            "metric"
        ]
        ==
        "qwk"
    ]
    .iloc[0]
)

qwk_ci_includes_zero = bool(
    qwk_pair_row[
        "difference_ci_low"
    ]
    <=
    0
    <=
    qwk_pair_row[
        "difference_ci_high"
    ]
)

locked_backbone_record = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "protocol_version": (
        PROTOCOL_VERSION
    ),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "locked_representative_backbone": (
        BACKBONE_A
    ),

    "locked_configuration": (
        BACKBONE_CONFIGS[
            BACKBONE_A
        ]
    ),

    "second_ranked_backbone": (
        BACKBONE_B
    ),

    "primary_selection_rule": (
        "Mean pooled OOF patient QWK "
        "across three seeds"
    ),

    "paired_qwk_difference": float(
        qwk_pair_row[
            "point_difference"
        ]
    ),

    "paired_qwk_difference_95_ci": [
        float(
            qwk_pair_row[
                "difference_ci_low"
            ]
        ),

        float(
            qwk_pair_row[
                "difference_ci_high"
            ]
        ),
    ],

    "paired_qwk_ci_includes_zero": (
        qwk_ci_includes_zero
    ),

    "interpretation": (
        "Backbone selection follows the frozen "
        "development-only ranking hierarchy. "
        "If the paired QWK interval includes zero, "
        "the selected backbone is a procedural "
        "selection rather than evidence of clear "
        "statistical superiority."
    ),

    "seed_ensemble_used_for_selection": False,

    "primary_patient_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "patient_aggregation": (
        LOCKED_AGGREGATION_RULE
    ),

    "official_test_loaded": False,

    "official_test_evaluated": False,
}

locked_backbone_path = (
    OUTPUT_ROOT
    / "locked_representative_backbone.json"
)

atomic_json_save(
    locked_backbone_record,
    locked_backbone_path,
)


# ============================================================
# 36. Decision summary
# ============================================================

decision_summary = {
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "backbones_compared": (
        BACKBONES_TO_RUN
    ),

    "seeds": (
        CONFIRMATION_SEEDS
    ),

    "selected_backbone": (
        BACKBONE_A
    ),

    "second_backbone": (
        BACKBONE_B
    ),

    "paired_qwk_ci_includes_zero": (
        qwk_ci_includes_zero
    ),

    "primary_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "aggregation": (
        LOCKED_AGGREGATION_RULE
    ),

    "official_test_loaded": False,

    "official_test_evaluated": False,

    "convnext_maxvit_note": (
        "ConvNeXt and MaxViT are intentionally absent "
        "from this benchmark because revised "
        "development-only loss/configuration selection "
        "has not yet been reconstructed for those "
        "backbones."
    ),
}

decision_summary_path = (
    OUTPUT_ROOT
    / "backbone_benchmark_decision_summary.json"
)

atomic_json_save(
    decision_summary,
    decision_summary_path,
)


# ============================================================
# 37. Display final results
# ============================================================

print(
    "\n"
    +
    "=" * 125
)

print(
    "PER-SEED UNIFORM BACKBONE BENCHMARK"
)

print(
    "=" * 125
)

display(
    per_seed_df[
        [
            "backbone_name",
            "model_name",
            "representative_configuration",
            "training_seed",

            "oof_patient_qwk",
            "oof_patient_qwk_ci_low",
            "oof_patient_qwk_ci_high",

            "oof_patient_mae",
            "oof_patient_macro_f1",
            "oof_patient_accuracy",

            "oof_image_qwk",
            "oof_image_mae",
        ]
    ]
    .sort_values(
        [
            "backbone_name",
            "training_seed",
        ]
    )
)


print(
    "\n"
    +
    "=" * 125
)

print(
    "THREE-SEED BACKBONE MEAN ± SD"
)

print(
    "=" * 125
)

display(
    ranked_df
)


print(
    "\n"
    +
    "=" * 125
)

print(
    "PAIRED PATIENT BOOTSTRAP: "
    "MEAN METRICS ACROSS SEEDS"
)

print(
    "=" * 125
)

display(
    paired_summary_df
)


print(
    "\n"
    +
    "=" * 125
)

print(
    "SEED-ENSEMBLE RESULTS — SUPPLEMENTARY"
)

print(
    "=" * 125
)

display(
    seed_ensemble_df
)


print(
    "\n"
    +
    "=" * 125
)

print(
    "LOCKED REPRESENTATIVE BACKBONE"
)

print(
    "=" * 125
)

print(
    "Selected backbone:",
    BACKBONE_A,
)

print(
    "Configuration:",
    BACKBONE_CONFIGS[
        BACKBONE_A
    ][
        "representative_configuration"
    ],
)

print(
    "Second-ranked backbone:",
    BACKBONE_B,
)

print(
    "Paired QWK CI includes zero:",
    qwk_ci_includes_zero,
)

print(
    "Seed ensemble used for selection:",
    False,
)


# ============================================================
# 38. Final status
# ============================================================

print(
    "\n"
    +
    "=" * 125
)

print(
    "06_backbone_benchmark_development_only COMPLETED"
)

print(
    "=" * 125
)

print(
    "Output directory:",
    OUTPUT_ROOT,
)

print(
    "Frozen analysis plan:",
    analysis_plan_path,
)

print(
    "Per-seed results:",
    per_seed_path,
)

print(
    "Multiseed summary:",
    summary_path,
)

print(
    "Backbone ranking:",
    ranking_path,
)

print(
    "Paired bootstrap:",
    paired_summary_path,
)

print(
    "Seed-ensemble results:",
    seed_ensemble_path,
)

print(
    "Locked backbone:",
    locked_backbone_path,
)

print(
    "Decision summary:",
    decision_summary_path,
)

print(
    "\nPrimary patient reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Locked patient aggregation:",
    LOCKED_AGGREGATION_RULE,
)

print(
    "Official LIMUC test data loaded:",
    False,
)

print(
    "Official LIMUC test data evaluated:",
    False,
)

print(
    "\nIMPORTANT:"
)

print(
    "ConvNeXt and MaxViT have NOT been included because "
    "their revised development-only configurations have "
    "not yet been frozen."
)

print(
    "Do not use the old test-guided ConvNeXt/MaxViT "
    "rankings in the revised manuscript."
)